# Exploration de la clé de jointure F5_2 ↔ F6_1

## Contexte

Pour calculer l'**intensité carbonique** (CO2 émis / énergie consommée), nous avons besoin de relier :
- **F1_4** : émissions CO2 par site (Facility)
- **F5_2** : consommation énergétique par LCP (Large Combustion Plant)
- **F6_1** : pont entre Installation et Facility

La hiérarchie E-PRTR est la suivante :
```
FACILITY  →  INSTALLATION  →  LCP (Part)
  F1_4          F6_1            F5_2
```

**Problème** : F5_2 ne contient pas de référence directe vers F6_1 ou F1_4.  
Il faut trouver une clé de jointure indirecte.

---

## 1. Setup & connexion

In [1]:
import sys
from pathlib import Path

# Ajoute la racine du projet au path Python
sys.path.insert(0, str(Path().resolve().parent))

import pandas as pd
from src.utils.db import connect_to_db

engine = connect_to_db()

parameters of DATABASE sucessfuly loaded 
connection sucessful


## 2. Chargement des données Belgique

On filtre directement en SQL pour ne charger que les lignes belges — pas besoin de charger les 400k+ lignes européennes en mémoire.

In [2]:
query_f5 = """
    SELECT * FROM bronze.f5_2_lcp_energy_emissions
    WHERE "countryName" = 'Belgium'
"""

query_f6 = """
    SELECT * FROM bronze.f6_1_ied_installations
    WHERE "CountryName" = 'Belgium'
"""

df_f5 = pd.read_sql(query_f5, con=engine)
df_f6 = pd.read_sql(query_f6, con=engine)

print(f"F5_2 Belgique : {df_f5.shape}")
print(f"F6_1 Belgique : {df_f6.shape}")

F5_2 Belgique : (10521, 14)
F6_1 Belgique : (20129, 31)


## 3. Exploration des formats d'ID

Première étape : regarder les valeurs brutes des colonnes d'ID pour identifier les patterns.

In [3]:
print("=== F5_2 — LCPInspireId (échantillon) ===")
print(df_f5['LCPInspireId'].unique()[:10].tolist())

print("\n=== F6_1 — InstallationInspireId (échantillon) ===")
print(df_f6['InstallationInspireId'].unique()[:10].tolist())

print("\n=== F6_1 — parent_facilityInspireId (échantillon) ===")
print(df_f6['parent_facilityInspireId'].unique()[:10].tolist())

=== F5_2 — LCPInspireId (échantillon) ===
['BE.WA/118010101.PART', 'https://data.ied_registry.omgeving.vlaanderen.be/id/productioninstallationpart/BE.VL.LCP000026.PART', 'BE.WA/114010101.PART', 'https://data.ied_registry.omgeving.vlaanderen.be/id/productioninstallationpart/BE.VL.LCP000039.PART', 'https://data.ied_registry.omgeving.vlaanderen.be/id/productioninstallationpart/BE.VL.LCP000058.PART', 'https://data.ied_registry.omgeving.vlaanderen.be/id/productioninstallationpart/BE.VL.LCP000024.PART', 'https://data.ied_registry.omgeving.vlaanderen.be/id/productioninstallationpart/BE.VL.LCP000059.PART', 'https://data.ied_registry.omgeving.vlaanderen.be/id/productioninstallationpart/BE.VL.LCP000051.PART', 'https://data.ied_registry.omgeving.vlaanderen.be/id/productioninstallationpart/BE.VL.LCP000065.PART', 'https://data.ied_registry.omgeving.vlaanderen.be/id/productioninstallationpart/BE.VL.LCP000011.PART']

=== F6_1 — InstallationInspireId (échantillon) ===
['https://data.ied_registry.omgev

## 4. Identification des 3 patterns d'ID

L'analyse visuelle révèle **3 formats distincts** selon la région :

| Région | Format | Exemple |
|--------|--------|---------|
| Wallonie | `BE.WA/XXXXXXXXX.SUFFIX` | `BE.WA/118010101.PART` |
| Flandre | URL + `BE.VL.LCPXXXXXX.SUFFIX` | `https://...BE.VL.LCP000026.PART` |
| EEA (ancien) | `BE.EEA/XXXXXX.SUFFIX` | `BE.EEA/BE0022.PART` |

In [4]:
# Segmentation par région — F5_2
f5_wa = df_f5[df_f5['LCPInspireId'].str.startswith('BE.WA')]
f5_vl = df_f5[df_f5['LCPInspireId'].str.startswith('https')]
f5_eea = df_f5[df_f5['LCPInspireId'].str.startswith('BE.EEA')]

print(f"Wallonie  : {len(f5_wa)} lignes")
print(f"Flandre   : {len(f5_vl)} lignes")
print(f"EEA       : {len(f5_eea)} lignes")
print(f"Total     : {len(df_f5)} lignes")

Wallonie  : 1889 lignes
Flandre   : 8359 lignes
EEA       : 117 lignes
Total     : 10521 lignes


## 5. Tentative de jointure — Wallonie

### Hypothèse
Les IDs Wallonie partagent un **numéro de site** commun malgré des suffixes différents :
```
BE.WA/047010000.FACILITY      ← Facility (F1_4 / F6_1)
BE.WA/047010200.INSTALLATION  ← Installation (F6_1)
BE.WA/047010101.PART          ← LCP (F5_2)
```
La clé de jointure = **numéro complet** entre `BE.WA/` et le suffixe `.XXXXX`

In [5]:
# Extraction de la clé numérique via regex
df_f5['site_key'] = df_f5['LCPInspireId'].str.extract(r'BE\.WA/(\d+)\.')
df_f6['site_key'] = df_f6['parent_facilityInspireId'].str.extract(r'BE\.WA/(\d+)\.')

print("Clés F5_2 (WA) :")
print(sorted(df_f5['site_key'].dropna().unique().tolist()))

print("\nClés F6_1 (WA) — échantillon :")
print(sorted(df_f6['site_key'].dropna().unique().tolist())[:15])

Clés F5_2 (WA) :
['000002514', '000004363', '047010101', '049010701', '049010702', '049010703', '095010101', '098010101', '103010101', '103010102', '112010101', '112010102', '113010101', '114010101', '115010101', '116010101', '117010101', '118010101', '119010101', '120010101', '189010201', '189010202', '189010203', '215010101', '225010101']

Clés F6_1 (WA) — échantillon :
['000000508', '000000632', '000000923', '000001048', '000001302', '000001308', '000001526', '000001769', '000001932', '000002421', '000002463', '000002859', '000003166', '000003350', '000003695']


### Réduction à 5 chiffres significatifs

Le numéro LCP `047010101` contient le code site `04701` + le sous-numéro d'installation.  
La Facility correspondante est `047010000` → préfixe `04701` commun.

In [6]:
# Réduction aux 5 premiers chiffres
df_f5['site_key_5'] = df_f5['site_key'].str[:5]
df_f6['site_key_5'] = df_f6['site_key'].str[:5]

keys_f5 = set(df_f5['site_key_5'].dropna())
keys_f6 = set(df_f6['site_key_5'].dropna())

print(f"Clés F5_2 (WA) : {len(keys_f5)}")
print(f"Clés F6_1 (WA) : {len(keys_f6)}")
print(f"Clés communes  : {len(keys_f5 & keys_f6)}")
print(f"Clés F5 sans match : {keys_f5 - keys_f6}")

Clés F5_2 (WA) : 18
Clés F6_1 (WA) : 337
Clés communes  : 18
Clés F5 sans match : set()


### Résultat jointure Wallonie

✅ **18 clés communes** entre F5_2 et F6_1 pour la Wallonie  
⚠️ 2 IDs exclus (`000002514`, `000004363`) — pas de Facility correspondante dans F6_1, documentés comme limitation

In [ ]:
# Jointure test Wallonie
df_joined_wa = df_f5.merge(
    df_f6[['site_key_5', 'parent_facilityInspireId', 'InstallationInspireId',
           'installationName', 'IEDMainActivityName']].drop_duplicates('site_key_5'),
    on='site_key_5',
    how='inner'
)

print(f"Lignes après jointure Wallonie : {len(df_joined_wa)}")
print(f"Sites uniques matchés : {df_joined_wa['site_key_5'].nunique()}")
df_joined_wa[['LCPInspireId', 'installationPartName', 'parent_facilityInspireId', 'installationName']].head(5)

Lignes après jointure Wallonie : 10521
Sites uniques matchés : 18


,LCPInspireId,installationPartName,parent_facilityInspireId,installationName
0,BE.WA/118010101.PART,Turbine à gaz,BE.WA/118010000.FACILITY,Combustion
1,https://data.ied_registry.omgeving.vlaanderen....,ELECTRABEL TURBOJET ZEDELGEM,https://data.ied_registry.omgeving.vlaanderen....,CONFIDENTIAL
2,BE.WA/114010101.PART,TURBINNE A GAZ (Saint-Ghislain I),BE.WA/114010000.FACILITY,Combustion
3,https://data.ied_registry.omgeving.vlaanderen....,INEOS PHENOL BELGIUM_LCP 2,https://data.ied_registry.omgeving.vlaanderen....,CONFIDENTIAL
4,https://data.ied_registry.omgeving.vlaanderen....,TOTALENERGIES OLEFINS ANTWERP_LCP 2,https://data.ied_registry.omgeving.vlaanderen....,CONFIDENTIAL


## 6. Problème Flandre — Jointure impossible par clé

### Constat
Les IDs Flandre ont des formats **incompatibles** entre F5_2 et F6_1 :

```
F5_2 LCPInspireId      : .../BE.VL.LCP000026.PART      → numéro LCP : 000026
F6_1 facilityInspireId : .../BE.VL.000001626.FACILITY  → numéro Facility : 000001626
```

Ces numéros sont indépendants — `000026` ≠ `000001626`. Aucune jointure par clé n'est possible.

In [8]:
# Démonstration du problème Flandre
f5_vl_keys = df_f5[df_f5['LCPInspireId'].str.startswith('https')]['LCPInspireId'].str.extract(r'BE\.VL\.LCP(\d+)')
f6_vl_keys = df_f6[df_f6['parent_facilityInspireId'].str.startswith('https')]['parent_facilityInspireId'].str.extract(r'BE\.VL\.(\d+)')

print("Numéros LCP Flandre (F5_2) :")
print(sorted(f5_vl_keys[0].dropna().unique().tolist())[:10])

print("\nNuméros Facility Flandre (F6_1) :")
print(sorted(f6_vl_keys[0].dropna().unique().tolist())[:10])

print("\n→ Formats incompatibles : jointure par clé impossible pour la Flandre")

Numéros LCP Flandre (F5_2) :
['000001', '000002', '000003', '000004', '000005', '000006', '000007', '000008', '000009', '000010']

Numéros Facility Flandre (F6_1) :
['000000001', '000000002', '000000003', '000000004', '000000005', '000000006', '000000007', '000000008', '000000011', '000000012']

→ Formats incompatibles : jointure par clé impossible pour la Flandre


In [ ]:
#nombre de LCP en FLANDRES
df_f5[df_f5['LCPInspireId'].str.startswith('https')]['LCPInspireId'].nunique()

79

In [ ]:
#Nombre de facility en FLANDRES
df_f6[df_f6['parent_facilityInspireId'].str.startswith('https')]['parent_facilityInspireId'].nunique()

1942

## 7. Pistes de solution pour la Flandre

### Option A — Jointure spatiale GPS (V2)
Les deux tables contiennent `Longitude` et `Latitude`. Un LCP et sa Facility sont sur le même site physique.  
→ Utiliser `geopy.distance.geodesic` ou `sklearn.BallTree` pour trouver les paires à moins de X mètres. 

→**78 sites LCP × 1942 facilities** → jointure spatiale faisable en V1 ?


### Option B — Jointure par nom de ville + nom de site (V2)
Rapprocher `City_Of_Facility` + `installationPartName` (F5_2) avec `City_of_Facility` + `installationName` (F6_1) via fuzzy matching.

---

## 8. Décision V1

| Région | Stratégie V1 | Lignes joinables |
|--------|-------------|------------------|
| **Wallonie** | Jointure par clé numérique (5 chiffres) | ✅ 18 sites |
| **Flandre** | Reporté en V2 — jointure spatiale GPS | ⏳ Flandre : 78 sites LCP × 1942 facilities|
| **EEA** | Exclu — format non supporté dans F6_1 | ❌ |

**Prochaine étape** : construire la table Silver `silver.intensite_carbonique_wa` sur les 18 sites Wallonie joinables.